In [4]:
import networkx as nx
import csv
import tkinter as tk
from tkinter import ttk, messagebox
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# -----------------------------
# Colors
# -----------------------------
BG_COLOR = "#ffe6f0"
SIDEBAR_COLOR = "#ffb6c1"
BTN_COLOR = "#ff69b4"
TEXT_COLOR = "#4a004a"

# -----------------------------
# Load Graph
# -----------------------------
def load_graph():
    G = nx.Graph()
    try:
        with open('first (2).csv', mode='r', encoding='latin-1') as file:
            reader = csv.reader(file, delimiter=';')
            for row in reader:
                if len(row) >= 2:
                    G.add_edge(row[0].strip(), row[1].strip())
    except:
        messagebox.showerror("Error", "CSV file not found!")
    return G

G = load_graph()

# -----------------------------
# Hierarchy Layout 
# -----------------------------
def hierarchy_pos(G, root, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5, visited=None):
    if visited is None:
        visited = set()

    visited.add(root)
    pos = {root: (xcenter, vert_loc)}

    neighbors = [n for n in G.neighbors(root) if n not in visited]

    if neighbors:
        dx = width / len(neighbors)
        nextx = xcenter - width/2 - dx/2

        for neighbor in neighbors:
            nextx += dx
            pos.update(hierarchy_pos(
                G,
                neighbor,
                width=dx,
                vert_gap=vert_gap,
                vert_loc=vert_loc - vert_gap,
                xcenter=nextx,
                visited=visited
            ))
    return pos

# -----------------------------
# Clear Graph Area
# -----------------------------
def clear_graph():
    for widget in graph_frame.winfo_children():
        widget.destroy()

# -----------------------------
# Draw Full Graph
# -----------------------------
def draw_full_graph():
    clear_graph()

    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(G, k=0.5)

    nx.draw(G, pos,
            with_labels=True,
            node_color="pink",
            edge_color="gray",
            node_size=900,
            font_size=7,
            ax=ax)

    canvas = FigureCanvasTkAgg(fig, master=graph_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

# -----------------------------
# Draw BFS Tree (HIERARCHICAL)
# -----------------------------
def draw_tree():
    root_name = entry_name1.get().strip()

    if root_name not in G:
        messagebox.showerror("Error", "Name not found!")
        return

    clear_graph()

    T = nx.bfs_tree(G, root_name)

    pos = hierarchy_pos(
        T,
        root_name,
        width=3,
        vert_gap=0.6
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    nx.draw(T, pos,
            with_labels=True,
            node_color="lightpink",
            edge_color="gray",
            node_size=700,
            font_size=6,
            font_weight='bold',
            ax=ax)

    canvas = FigureCanvasTkAgg(fig, master=graph_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

# -----------------------------
# Shortest Path
# -----------------------------
def draw_path():
    p1 = entry_name1.get().strip()
    p2 = entry_name2.get().strip()

    if p1 not in G or p2 not in G:
        messagebox.showerror("Error", "Names not found!")
        return

    try:
        path = nx.shortest_path(G, source=p1, target=p2)

        clear_graph()

        subG = G.subgraph(path)
        fig, ax = plt.subplots(figsize=(8, 6))
        pos = nx.spring_layout(subG)

        nx.draw(subG, pos,
                with_labels=True,
                node_color="lightblue",
                edge_color="black",
                node_size=1200,
                font_size=8,
                ax=ax)

        canvas = FigureCanvasTkAgg(fig, master=graph_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        messagebox.showinfo("Path",
                            " → ".join(path) +
                            f"\nDegree: {len(path)-1}")

    except:
        messagebox.showerror("Error", "No path found!")

# -----------------------------
# Mutual Friends
# -----------------------------
def show_mutuals():
    p1 = entry_name1.get().strip()
    p2 = entry_name2.get().strip()

    if p1 not in G or p2 not in G:
        messagebox.showerror("Error", "Names not found!")
        return

    mutual = list(set(G.neighbors(p1)) & set(G.neighbors(p2)))

    messagebox.showinfo("Mutual Friends",
                        ", ".join(mutual) if mutual else "None")

# -----------------------------
# Top 5 Friends
# -----------------------------
def top_5_friends():
    if len(G.nodes()) == 0:
        messagebox.showerror("Error", "Graph is empty!")
        return

    sorted_users = sorted(G.nodes(), key=lambda x: G.degree(x), reverse=True)
    top5 = sorted_users[:5]

    result = "Top 5 Most Connected Users:\n\n"
    for i, user in enumerate(top5, 1):
        result += f"{i}. {user} ({G.degree(user)} friends)\n"

    messagebox.showinfo("Top Users", result)

# -----------------------------
# GUI
# -----------------------------
root = tk.Tk()
root.title("Social Network Analyzer")
root.geometry("900x600")
root.configure(bg=BG_COLOR)

style = ttk.Style()
style.theme_use('clam')

style.configure("TButton",
                font=("Arial", 10, "bold"),
                background=BTN_COLOR,
                foreground=TEXT_COLOR)

style.map("TButton",
          background=[('active', SIDEBAR_COLOR)])

style.configure("TLabel",
                background=SIDEBAR_COLOR,
                foreground=TEXT_COLOR)

# Layout
left_frame = tk.Frame(root, width=300, bg=SIDEBAR_COLOR, padx=10, pady=10)
left_frame.pack(side=tk.LEFT, fill=tk.Y)

graph_frame = tk.Frame(root, bg="white")
graph_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

# Inputs
tk.Label(left_frame, text="Control Panel",
         font=("Arial", 16, "bold"),
         bg=SIDEBAR_COLOR,
         fg=TEXT_COLOR).pack(pady=10)

tk.Label(left_frame, text="Root:",
         bg=SIDEBAR_COLOR,
         fg=TEXT_COLOR).pack(anchor="w")

entry_name1 = tk.Entry(left_frame)
entry_name1.pack(pady=5)

tk.Label(left_frame, text="Target:",
         bg=SIDEBAR_COLOR,
         fg=TEXT_COLOR).pack(anchor="w")

entry_name2 = tk.Entry(left_frame)
entry_name2.pack(pady=5)

ttk.Separator(left_frame).pack(fill='x', pady=10)

# Buttons
ttk.Button(left_frame, text="Show Graph", command=draw_full_graph).pack(pady=5)
ttk.Button(left_frame, text="BFS Tree", command=draw_tree).pack(pady=5)
ttk.Button(left_frame, text="Shortest Path", command=draw_path).pack(pady=5)
ttk.Button(left_frame, text="Mutual Friends", command=show_mutuals).pack(pady=5)
ttk.Button(left_frame, text="Top 5 Friends", command=top_5_friends).pack(pady=5)

# Node List
listbox = tk.Listbox(left_frame, height=10)
listbox.pack(pady=10)

for node in G.nodes():
    listbox.insert(tk.END, node)

# Initial Message
tk.Label(graph_frame,
         text="Graph will appear here",
         bg="white").pack()

root.mainloop()